[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ushaloy/PINN-LEWS-HYDROL86065/blob/main/notebooks/03_baselines_external_analysis.ipynb)

# Baselines, external sites and strength sensitivity — HYDROL86065

This notebook runs the Python scripts of the repository, in order. It does **not** replace notebooks 01 and 02.

| Step | Scripts | Produces | Time |
|---|---|---|---|
| 2 | `baselines/` | persistence, ARX, random forest, gradient boosting, Richards solver (Table 3, Table 5, Tables S7–S8) → `results/baselines/` | ~15 min |
| 3 | `analysis/fos_strength_sensitivity.py` | FoS vs the assumed c′ and φ′ (Table S12) → `results/chrono/fos_sensitivity_strength.csv` | seconds |
| 4 | `external_sites/` | transfer to Utuado and California (Table 6, Tables S10–S11) → `results/external/` | hours (full) |

Set `QUICK = True` in step 1 for a few-minute pipeline check of step 4 (the numbers are not results). Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

## 1. Get the repository and choose the mode

In [ ]:
import os, subprocess, sys

QUICK = True     # True: pipeline check of step 4 (numbers are not results). False: full external-site runs.

REPO_NAME = "PINN-LEWS-HYDROL86065"
REPO_URL  = "https://github.com/Ushaloy/PINN-LEWS-HYDROL86065"
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = os.path.join("/content", REPO_NAME)
    if not os.path.isdir(os.path.join(REPO, "data")):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
else:   # run from the repository's notebooks/ folder
    here = os.getcwd()
    REPO = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
assert os.path.isdir(os.path.join(REPO, "data")), "data/ not found: run this notebook inside the repository"
os.chdir(REPO)
print("repository:", REPO)
Q = "--quick" if QUICK else ""

## 2. Baselines (CPU, ~15 min)
The commands of `baselines/README.md`. Results: `results/baselines/`.

In [ ]:
%%bash
cd baselines
python prep_features.py chrono
python prep_features.py season
python run_ml.py
python run_phys.py chrono free
python run_phys.py chrono onestep
python run_phys.py season free
python run_phys.py season onestep
python run_transfer_ml.py
ls ../results/baselines

## 3. FoS sensitivity to the assumed strength parameters (seconds)
Reads `results/chrono/fos_timeseries_main.csv`; c′ and φ′ were assumed, not measured.

In [ ]:
!python analysis/fos_strength_sensitivity.py results/chrono

## 4. External sites (Utuado, California)
Results: `results/external/`. The reference results of the manuscript are in `results/external/reference/`.
With `QUICK = False` each run takes about 30–40 min; run the cells one at a time.

In [ ]:
!python external_sites/run_external.py transfer --site utuado --soil measured $Q
!python external_sites/run_external.py transfer --site utuado --soil range $Q
!python external_sites/run_external.py transfer --site utuado --soil dev108 $Q
!python external_sites/run_external.py transfer --site california --soil measured $Q

In [ ]:
!python external_sites/run_external.py local --site utuado --seeds 42,43 $Q
!python external_sites/run_external.py local --site california --seeds 42,43 $Q
!python external_sites/baselines_external.py --site utuado
!python external_sites/baselines_external.py --site california

## 5. Keep the results (Colab only)
The Colab disk is erased when the session ends. This copies the new results to Google Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/{REPO_NAME}_results/scripts"
    for sub in ("baselines", "external"):
        if os.path.isdir(f"results/{sub}"):
            shutil.copytree(f"results/{sub}", f"{dst}/{sub}", dirs_exist_ok=True)
    shutil.copy("results/chrono/fos_sensitivity_strength.csv", dst)
    print("copied to", dst)